## Setup

In [1]:
from typing import Any, Literal
from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pydantic import BaseModel, Field
from sglang import (
    function,
    system,
    user,
    assistant,
    gen,
    set_default_backend,
    RuntimeEndpoint,
)

tqdm.pandas()  # load tqdm's pandas support
pd.set_option("display.max_colwidth", None)

In [2]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=32, progress_bar=True, verbose=1)

In [3]:
repo_id = "meta-llama/Llama-3.2-3B-Instruct"

# from datasets import load_dataset
# from huggingface_hub import InferenceClient
# llm_client = InferenceClient(model=repo_id, timeout=120)
# llm_client.text_generation(prompt="How are you today? ", max_new_tokens=20)

print(
    f'Run:\n\npython -m sglang.launch_server --model-path {repo_id} --port 30000 --host 0.0.0.0\n\n... and wait for "The server is fired up and ready to roll!" to appear.'
)

Run:

python -m sglang.launch_server --model-path meta-llama/Llama-3.2-3B-Instruct --port 30000 --host 0.0.0.0

... and wait for "The server is fired up and ready to roll!" to appear.


In [4]:
set_default_backend(RuntimeEndpoint("http://localhost:30000"))

## Prepare the data

In [5]:
base = Path(".").resolve() / "eval-all" / "data"
assert base.is_dir() and base.exists()
base


PosixPath('/workspaces/ts-qa/eval-all/data')

In [6]:
df = pd.read_hdf(base / "llmjudge_eval_set.h5", key="data")
df.head()

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text
2635863,4.2.2,T5 Base,full,26172,3,kicking a ball,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['throwing a ball', 'dancing', 'waving', 'sitting down'], 'action_sentence': ['The person is starting off the sequence with throwing a ball', 'The person is engaging in dancing', 'The person is performing waving', 'Finally, the person is sitting down']}","The person is starting off the sequence with throwing a ball from 0:0.0 to 0:4.0 (for 4.0 seconds). The person is engaging in dancing. This action is starting at 0:4.0 and happens until 0:8.0, meaning the engagement in this is for 4.0 seconds. The person is performing waving. This is happening from 0:8.0 to 0:12.0, which means for a total time of 4.0 seconds. Finally, the person is sitting down. This is happening from 0:12.0 to 0:16.0, which means for a total time of 4.0 seconds.",last_open,Which activity can be identified as the very last action carried out by the person?,open,We see someone who is sitting down.
1773964,4.2.1,MVP,50,24792,4,sh the a with,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['golfing (swinging a club)', 'waving', 'waving', 'waving'], 'action_sentence': ['As the first action, the person is golfing (swinging a club)', 'Someone is waving', 'The person is waving', 'Finally, the person is waving']}","As the first action, the person is golfing (swinging a club). This is happening from 0:0.0 to 0:4.0, which means for a total time of 4.0 seconds. Someone is waving from 0:4.0 to 0:8.0 (4.0 seconds total). The person is waving. It is starting at second 0:8.0 and continuing until 0:12.0, so the person is doing this for 4.0 seconds. Finally, the person is waving. This is happening from 0:12.0 to 0:16.0, which means for a total time of 4.0 seconds.",last_open,What action can be identified as the final one performed?,open,The person is waving.
1314002,4.2.1,Llama 3 (8B),full,25800,2,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['picking something up with both hands', 'picking something up with both hands', 'shaking hands', 'playing guitar'], 'action_sentence': ['The activity picking something up with both hands is the first action done by the person', 'The activity picking something up with both hands can be observed', 'The movement is shaking hands', 'The last movement the person is doing is playing guitar']}","The activity picking something up with both hands is the first action done by the person. It is starting at second 0:0.0 and continuing until 0:4.0, so the person is doing this for 4.0 seconds. The activity picking something up with both hands can be observed from 0:4.0 to 0:8.0 (for 4.0 seconds). The movement is shaking hands. This action is starting at 0:8.0 and continues until 0:12.0, resulting in a total time of 4.0 seconds. The last movement the person is doing is playing guitar. It is starting at second 0:12.0 and continuing until 0:16.0, so the person is doing this for 4.0 seconds.",count_open,How many occurrences of picking something up with both hands have taken place involving the person?,open,1
2502191,4.2.2,Falcon 2 (11B),full,26438,1,jumping once,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['sitting down', 'sitting down', 'jumping once', 'kicking a ball'], 'action_sentence': ['The person is starting off the sequence with sitting down', 'The person is sitting down', 'Someone is jumping once', 'Kicking a ball is the last action performed by the person']}","The person is starting off the sequence with sitting down. This action is starting at 0:0.0 and happens until 0:4.0, meaning the engagement in this is for 4.0 seconds. The person is sitting down. This is happening from 0:4.0 to 0:8.0, which means for a total time of 4.0 seconds. Someone is jumping once. This action is starting at 0:8.0 and c

In [7]:
# Duplicate the dataframe 10 times
df = pd.concat([df] * 50, ignore_index=True)
df.shape

(10000, 12)

In [8]:
JUDGE_PROMPT_TEMPLATE = """You will be given a scene_description with timesamps and a scene_question regarding that scene. You will also be given a reference_answer and system_answer couple.
Your task is to provide a TotalRating scoring how well the system_answer answers the user concerns expressed in the scene_question. The reference_answer is provided as reference for a very good answer.
Give your answer on a scale of 1 to 4, where 1 means that the system_answer is not helpful at all, and 4 means that the system_answer completely and helpfully addresses the scene_question.
You will also provide a brief rationale for your rating.

Here is the scale you should use to build your answer:
1: The system_answer is terrible: completely irrelevant to the question asked, or corrupt in some way (e.g., empty or nonsensical)
2: The system_answer is bad: misses some key aspects of the question or is factually incorrect
3: The system_answer is good: correctly answers the scene_question and is helpful in the context of the scene

Provide your feedback in the following JSON format:
{{
    "BriefRationale:": "<your rationale, at most 3 sentences>",
    "TotalRating:": <your rating>
}}

You MUST provide values for "BriefRationale" and "TotalRating" in your answer.
Do NOT respond with any other text in your answer.

Now here are the question and answer:

- scene_description: {scene_description}
- scene_question: {scene_question}
- reference_answer: {reference_answer}
- system_answer: {system_answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
"""


max_rationale = 500


class JudgePrompt(BaseModel):
    BriefRationale: str = Field(
        ..., max_length=max_rationale, description="Your rationale, at most 3 sentences"
    )
    TotalRating: Literal[1, 2, 3] = Field(
        ...,
        description="Your rating on a scale of 1 to 4",
    )


@function
def sglang_llm_judge(s, row):
    # scene_description = TODO make table from this
    scene_description = row["textual_description"]
    judge_prompt = JUDGE_PROMPT_TEMPLATE.format(
        scene_description=scene_description,
        scene_question=row["question"],
        reference_answer=row["answer_text"],
        system_answer=row["prediction_text"],
    )  # + str(np.random.randint(0, 100000))

    s += user(judge_prompt)
    s += assistant(
        gen(
            "response",
            max_tokens=max_rationale + 100,
            temperature=0,
            # json_schema=JudgePrompt.model_json_schema(),
        )
    )


df["llm_judge_raw"] = df.parallel_apply(
    lambda x: (
        x["llm_judge_raw"]
        if "llm_judge_raw" in x and x["llm_judge_raw"] != pd.NA
        else sglang_llm_judge.run(
            row=x,
        )["response"]
    ),
    axis=1,
)


def decode(response) -> dict[str, Any]:
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return dict(BriefRationale=pd.NA, TotalRating=pd.NA)


df[["llm_judge_rationale", "llm_judge_score"]] = (
    df["llm_judge_raw"]
    .apply(decode)
    .apply(pd.Series)[["BriefRationale", "TotalRating"]]
)

# Adjust the score to be in the same range as the human scores
df["llm_judge_score"] *= 3

In [9]:
print("Correlation between LLM-as-a-judge and the human raters:")
print(f"{df['llm_judge_score'].corr(df['human_score'], method='pearson'):.3f}")

Correlation between LLM-as-a-judge and the human raters:


KeyError: 'human_score'